In [4]:
import numpy as np
from RobustWorstCasePortfolio.robust_mvo_solvers import robust_mvo_general_ellipsoid

T = 52
mu_hat = np.array([0.12, 0.10, 0.07])
Sigma_hat = np.array([[0.0064, 0.0008, 0.0011],
                      [0.0008, 0.0025, 0.0014],
                      [0.0011, 0.0014, 0.0040]])
lam = 2.0
epsilon = 0.1
kappa = 0.1
S_mu = Sigma_hat/T
P_shape = np.eye(3)

robust_mvo_general_ellipsoid(mu_hat, Sigma_hat, lam, epsilon, kappa, S_mu, P_shape)

array([0.38264444, 0.34828537, 0.26907018])

In [3]:
import numpy as np
import cvxpy as cp
import time
from scipy.linalg import sqrtm, inv

from robust_mvo_solvers import robust_mvo_general, robust_mvo_fast

def generate_dummy_data(N=10):
    """Generates valid random financial data."""
    np.random.seed(42)
    mu = np.random.uniform(0.05, 0.20, N)
    # Create random correlation matrix
    A = np.random.randn(N, N)
    Sigma = A @ A.T
    # Normalize to have realistic volatilities
    d = np.sqrt(np.diag(Sigma))
    Sigma = Sigma / np.outer(d, d) * 0.2  # 20% average vol
    # Ensure PSD
    Sigma += np.eye(N) * 1e-4
    return mu, Sigma

def run_tests():
    # 1. Setup Data
    N_ASSETS = 30  # Keep it small enough for the slow solver to finish quickly
    mu_hat, Sigma_hat = generate_dummy_data(N_ASSETS)
    
    # Parameters
    lam = 5.0
    epsilon = 0.5
    kappa = 1.0
    S_mu = Sigma_hat / 252.0  # Daily scaling

    print(f"{'='*80}")
    print(f"ROBUST MVO CONSISTENCY CHECK (N={N_ASSETS})")
    print(f"{'='*80}")
    print(f"{'View':<15} | {'General (SDP) Time':<20} | {'Fast (SOCP) Time':<20} | {'Diff':<10} | {'Match?'}")
    print("-" * 80)

    # 2. Define the Test Cases (Views)
    test_cases = [
        ("Spherical", "Identity"),
        ("Statistical", "Sigma"),
        ("Inverse", "Sigma_inv")
    ]

    for view_name, method in test_cases:
        
        # --- PREPARE INPUT MATRICES ---
        if method == "Identity":
            # Spherical: P=I, Q=I
            P_shape = np.eye(N_ASSETS)
            Q_cov   = np.eye(N_ASSETS)
            
        elif method == "Sigma":
            # Statistical: P=Sigma^0.5, Q=Sigma
            # Note: We must be careful with sqrtm for P
            P_shape = np.real(sqrtm(Sigma_hat))
            Q_cov   = Sigma_hat
            
        elif method == "Sigma_inv":
            # Inverse/Regularized: P=Sigma^-0.5, Q=Sigma^-1
            try:
                inv_Sigma = inv(Sigma_hat)
            except:
                inv_Sigma = np.linalg.pinv(Sigma_hat)
            
            P_shape = np.real(sqrtm(inv_Sigma))
            Q_cov   = inv_Sigma

        # --- RUN GENERAL (SLOW) SOLVER ---
        t0 = time.time()
        w_gen = robust_mvo_general(
            mu_hat, Sigma_hat, lam, epsilon, kappa, S_mu, P_shape
        )
        t_gen = time.time() - t0

        # --- RUN FAST (OPTIMIZED) SOLVER ---
        t1 = time.time()
        w_fast = robust_mvo_fast(
            mu_hat, Sigma_hat, lam, epsilon, kappa, Q_cov, S_mu
        )
        t_fast = time.time() - t1

        # --- COMPARE RESULTS ---
        # Calculate Euclidean distance between weight vectors
        diff = np.linalg.norm(w_gen - w_fast)
        is_match = "✅" if diff < 1e-4 else "❌"

        # Output row
        print(f"{view_name:<15} | {t_gen:.4f}s {' '*13} | {t_fast:.4f}s {' '*13} | {diff:.1e}  | {is_match}")

    print("-" * 80)
    print("\nInterpretation:")
    print("1. ✅ MATCH: Means the Fast Code is mathematically identical to the General SDP.")
    print("2. If 'Inverse' matches, it proves that Z (the dual variable) is indeed 0")
    print("   and the 'Condition Number' regularization logic holds.")

if __name__ == "__main__":
    run_tests()

ROBUST MVO CONSISTENCY CHECK (N=30)
View            | General (SDP) Time   | Fast (SOCP) Time     | Diff       | Match?
--------------------------------------------------------------------------------
Spherical       | 0.9836s               | 0.0332s               | 8.2e-07  | ✅
Statistical     | 2.7313s               | 0.0383s               | 2.8e-05  | ✅
Inverse         | 3.7253s               | 0.0230s               | 1.7e-06  | ✅
--------------------------------------------------------------------------------

Interpretation:
1. ✅ MATCH: Means the Fast Code is mathematically identical to the General SDP.
2. If 'Inverse' matches, it proves that Z (the dual variable) is indeed 0
   and the 'Condition Number' regularization logic holds.
